# EmotiContigo – Evaluación y ajuste fino del modelo de emociones

Este notebook hace dos cosas:

1. **Evalúa el modelo actual** (`abhilash88/face-emotion-detection`, ViT-Base entrenado con FER2013) sobre la parte de prueba original de FER2013 (PrivateTest).
2. **Lo ajusta (fine-tuning)** unas épocas más, con pesos por clase y aumentos de datos, y compara los resultados.

Al final guarda el modelo nuevo, las métricas y las matrices de confusión.

**Antes de empezar:** en Colab ve a *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

Tiempo aproximado con GPU T4: evaluación ~3 min, entrenamiento ~15–30 min.

## 1. Configuración

In [ ]:
# Cambia a True para una corrida rápida de prueba (pocas imágenes) antes de la real
MODO_PRUEBA = False

MODELO_BASE = "abhilash88/face-emotion-detection"
DATASET = "abhilash88/fer2013-enhanced"
EPOCAS = 3
TASA_APRENDIZAJE = 2e-5
LOTE = 64
SEMILLA = 42
CARPETA_SALIDA = "modelo_emoticontigo"
CARPETA_RESULTADOS = "resultados"

# Nombres en el mismo orden que las etiquetas de FER2013 (0 a 6) y que src/preprocess.py
EMOCIONES = ["enfado", "asco", "miedo", "felicidad", "tristeza", "sorpresa", "neutral"]

In [ ]:
!nvidia-smi -L || echo "Sin GPU: activa la GPU en Entorno de ejecución"
!pip install -q "transformers>=4.41,<5" "datasets>=2.14" "accelerate>=0.26" scikit-learn

In [ ]:
import json
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torchvision import transforms
from transformers import (
    Trainer, TrainingArguments, ViTForImageClassification, ViTImageProcessor, set_seed,
)

set_seed(SEMILLA)
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO)

## 2. Cargar FER2013

In [ ]:
from collections import Counter
from datasets import DatasetDict, concatenate_datasets

crudo = load_dataset(DATASET)
todo = concatenate_datasets([crudo[parte] for parte in crudo])
print("Valores de 'usage':", Counter(todo["usage"]))

# Particion ORIGINAL de FER2013 (columna usage), la misma con la que se suele
# entrenar y evaluar. Las particiones propias de este dataset mezclan imagenes que el
# modelo actual ya vio al entrenar, y eso inflaria sus resultados.
datos = DatasetDict({
    "train": todo.filter(lambda uso: uso == "Training", input_columns="usage"),
    "validation": todo.filter(lambda uso: uso == "PublicTest", input_columns="usage"),
    "test": todo.filter(lambda uso: uso == "PrivateTest", input_columns="usage"),
})
assert all(len(datos[parte]) > 0 for parte in datos), "Revisa los valores de 'usage'"
if MODO_PRUEBA:
    for parte in datos:
        datos[parte] = datos[parte].shuffle(seed=SEMILLA).select(range(300))
print(datos)

In [ ]:
def a_imagen(ejemplo):
    """Convierte la cadena de píxeles de FER2013 (48x48, gris) en imagen RGB."""
    pixeles = np.array(ejemplo["pixels"].split(), dtype=np.uint8).reshape(48, 48)
    return Image.fromarray(pixeles).convert("RGB")


def contar_clases(parte):
    conteo = np.bincount(parte["emotion"], minlength=len(EMOCIONES))
    return dict(zip(EMOCIONES, conteo.tolist()))


for parte in datos:
    print(parte, contar_clases(datos[parte]))

In [ ]:
# Muestra algunas imágenes de entrenamiento
fig, ejes = plt.subplots(2, 7, figsize=(14, 4.5))
for indice, emocion in enumerate(EMOCIONES):
    ejemplos = datos["train"].filter(lambda e: e["emotion"] == indice)
    for fila in range(2):
        ejes[fila, indice].axis("off")
        if fila < len(ejemplos):
            ejes[fila, indice].imshow(a_imagen(ejemplos[fila]))
    ejes[0, indice].set_title(emocion)
plt.tight_layout()
plt.show()

## 3. Preparar las imágenes para el modelo

In [ ]:
procesador = ViTImageProcessor.from_pretrained(MODELO_BASE)
TAMANO = procesador.size["height"]
normalizar = transforms.Normalize(mean=procesador.image_mean, std=procesador.image_std)

# Aumentos solo para entrenamiento: variaciones de encuadre, giro y luz
aumentos_entrenamiento = transforms.Compose([
    transforms.RandomResizedCrop(TAMANO, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    normalizar,
])
sin_aumentos = transforms.Compose([
    transforms.Resize((TAMANO, TAMANO)),
    transforms.ToTensor(),
    normalizar,
])


def preparar(transformacion):
    def aplicar(lote):
        imagenes = [a_imagen({"pixels": p}) for p in lote["pixels"]]
        return {
            "pixel_values": [transformacion(imagen) for imagen in imagenes],
            "labels": lote["emotion"],
        }
    return aplicar


entrenamiento = datos["train"].with_transform(preparar(aumentos_entrenamiento))
validacion = datos["validation"].with_transform(preparar(sin_aumentos))
prueba = datos["test"].with_transform(preparar(sin_aumentos))


def juntar(ejemplos):
    return {
        "pixel_values": torch.stack([e["pixel_values"] for e in ejemplos]),
        "labels": torch.tensor([e["labels"] for e in ejemplos]),
    }

## 4. Métricas

In [ ]:
def calcular_metricas(prediccion):
    etiquetas = prediccion.label_ids
    predichas = np.argmax(prediccion.predictions, axis=1)
    return {
        "exactitud": accuracy_score(etiquetas, predichas),
        "f1_macro": f1_score(etiquetas, predichas, average="macro"),
    }


def reporte(nombre, salida):
    etiquetas = salida.label_ids
    predichas = np.argmax(salida.predictions, axis=1)
    print(f"=== {nombre} ===")
    print(classification_report(etiquetas, predichas, labels=range(len(EMOCIONES)), target_names=EMOCIONES,
                                digits=3, zero_division=0))
    texto = classification_report(
        etiquetas, predichas, labels=range(len(EMOCIONES)), target_names=EMOCIONES,
        output_dict=True, zero_division=0,
    )
    matriz = confusion_matrix(etiquetas, predichas, labels=range(len(EMOCIONES)), normalize="true")
    return texto, matriz


def dibujar_matriz(matriz, titulo, archivo):
    fig, eje = plt.subplots(figsize=(7, 6))
    imagen = eje.imshow(matriz, cmap="Greens", vmin=0, vmax=1)
    eje.set_xticks(range(len(EMOCIONES)), EMOCIONES, rotation=45, ha="right")
    eje.set_yticks(range(len(EMOCIONES)), EMOCIONES)
    eje.set_xlabel("Predicha")
    eje.set_ylabel("Real")
    eje.set_title(titulo)
    for fila in range(len(EMOCIONES)):
        for columna in range(len(EMOCIONES)):
            valor = matriz[fila, columna]
            eje.text(columna, fila, f"{valor:.2f}", ha="center", va="center",
                     color="white" if valor > 0.5 else "black", fontsize=8)
    fig.colorbar(imagen)
    fig.tight_layout()
    fig.savefig(os.path.join(CARPETA_RESULTADOS, archivo), dpi=150)
    plt.show()

## 5. Evaluar el modelo actual (línea base)

Se mide sobre la parte de **prueba**, que no se usa para entrenar.

In [ ]:
modelo_base = ViTForImageClassification.from_pretrained(MODELO_BASE)
print("Etiquetas del modelo:", modelo_base.config.id2label)

argumentos_evaluacion = TrainingArguments(
    output_dir="tmp_evaluacion",
    per_device_eval_batch_size=LOTE,
    fp16=DISPOSITIVO == "cuda",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
)
evaluador = Trainer(
    model=modelo_base, args=argumentos_evaluacion,
    data_collator=juntar, compute_metrics=calcular_metricas,
)
salida_base = evaluador.predict(prueba)
print(salida_base.metrics)
reporte_base, matriz_base = reporte("Modelo actual", salida_base)
dibujar_matriz(matriz_base, "Modelo actual", "matriz_modelo_actual.png")

## 6. Ajuste fino

- Parte del modelo actual (no desde cero).
- **Pesos por clase**: las emociones con pocas imágenes (sobre todo *asco*) pesan más en el error.
- **Aumentos de datos** en entrenamiento (encuadre, giro, brillo y contraste).
- Se queda con la época que tenga mejor **F1 macro** en validación.

In [ ]:
conteo = np.bincount(datos["train"]["emotion"], minlength=len(EMOCIONES)).astype(float)
# Raíz del inverso de la frecuencia: compensa sin exagerar las clases pequeñas
pesos = np.sqrt(conteo.sum() / np.maximum(conteo, 1))
pesos = pesos / pesos.mean()
pesos_clase = torch.tensor(pesos, dtype=torch.float)
print(dict(zip(EMOCIONES, np.round(pesos, 2).tolist())))


class EntrenadorConPesos(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        etiquetas = inputs.pop("labels")
        salidas = model(**inputs)
        perdida = torch.nn.functional.cross_entropy(
            salidas.logits, etiquetas, weight=pesos_clase.to(salidas.logits.device),
        )
        return (perdida, salidas) if return_outputs else perdida

In [ ]:
modelo = ViTForImageClassification.from_pretrained(MODELO_BASE)
# Etiquetas LABEL_0..LABEL_6: src/model.py las traduce al español
modelo.config.id2label = {i: f"LABEL_{i}" for i in range(len(EMOCIONES))}
modelo.config.label2id = {f"LABEL_{i}": i for i in range(len(EMOCIONES))}

# 10 % de los pasos para el calentamiento (warmup_ratio ya no existe en transformers 5)
pasos_totales = math.ceil(len(datos["train"]) / LOTE) * EPOCAS
pasos_calentamiento = int(0.1 * pasos_totales)

argumentos = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCAS,
    learning_rate=TASA_APRENDIZAJE,
    per_device_train_batch_size=LOTE,
    per_device_eval_batch_size=LOTE,
    warmup_steps=pasos_calentamiento,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=DISPOSITIVO == "cuda",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    logging_steps=50,
    seed=SEMILLA,
    report_to="none",
)
entrenador = EntrenadorConPesos(
    model=modelo, args=argumentos,
    train_dataset=entrenamiento, eval_dataset=validacion,
    data_collator=juntar, compute_metrics=calcular_metricas,
)
entrenador.train()

## 7. Evaluar el modelo ajustado y comparar

In [ ]:
salida_nuevo = entrenador.predict(prueba)
print(salida_nuevo.metrics)
reporte_nuevo, matriz_nuevo = reporte("Modelo ajustado", salida_nuevo)
dibujar_matriz(matriz_nuevo, "Modelo ajustado", "matriz_modelo_ajustado.png")

In [ ]:
print(f"{'':12}{'actual':>10}{'ajustado':>10}{'diferencia':>12}")
for nombre, clave in [("Exactitud", "accuracy"), ("F1 macro", "macro avg")]:
    if clave == "accuracy":
        antes, despues = reporte_base["accuracy"], reporte_nuevo["accuracy"]
    else:
        antes, despues = reporte_base[clave]["f1-score"], reporte_nuevo[clave]["f1-score"]
    print(f"{nombre:12}{antes:10.3f}{despues:10.3f}{despues - antes:+12.3f}")
print()
print("F1 por emoción")
for emocion in EMOCIONES:
    antes = reporte_base[emocion]["f1-score"]
    despues = reporte_nuevo[emocion]["f1-score"]
    print(f"{emocion:12}{antes:10.3f}{despues:10.3f}{despues - antes:+12.3f}")

## 8. Guardar el modelo y los resultados

In [ ]:
entrenador.save_model(CARPETA_SALIDA)
procesador.save_pretrained(CARPETA_SALIDA)

resumen = {
    "modelo_base": MODELO_BASE,
    "dataset": DATASET,
    "modo_prueba": MODO_PRUEBA,
    "epocas": EPOCAS,
    "tasa_aprendizaje": TASA_APRENDIZAJE,
    "lote": LOTE,
    "pesos_clase": dict(zip(EMOCIONES, np.round(pesos, 4).tolist())),
    "imagenes": {parte: len(datos[parte]) for parte in datos},
    "modelo_actual": reporte_base,
    "modelo_ajustado": reporte_nuevo,
}
with open(os.path.join(CARPETA_RESULTADOS, "metricas.json"), "w", encoding="utf-8") as archivo:
    json.dump(resumen, archivo, ensure_ascii=False, indent=2)

!zip -qr modelo_emoticontigo.zip {CARPETA_SALIDA}
!zip -qr resultados.zip {CARPETA_RESULTADOS}
print("Listo: descarga modelo_emoticontigo.zip y resultados.zip")

In [ ]:
# Descarga los dos archivos a tu computadora
from google.colab import files
files.download("resultados.zip")
files.download("modelo_emoticontigo.zip")

## 9. Probar una imagen con el modelo nuevo

In [ ]:
from transformers import pipeline

clasificador = pipeline("image-classification", model=CARPETA_SALIDA, top_k=3,
                        device=0 if DISPOSITIVO == "cuda" else -1)
ejemplo = datos["test"][0]
print("Real:", EMOCIONES[ejemplo["emotion"]])
for resultado in clasificador(a_imagen(ejemplo)):
    indice = int(resultado["label"].split("_")[1])
    print(f"  {EMOCIONES[indice]:10} {resultado['score'] * 100:.2f} %")